# Data Exploration and Visualizations

*Exploratory Data Analysis (EDA)* is how a data scientist gets to know a dataset: where it came from, what it represents, which features describe it, what type of data each feature holds, and what stories come out of it.

<img src='https://d3lkc3n5th01x7.cloudfront.net/wp-content/uploads/2023/05/08040831/Exploratory-Data-Analysis.png'>

Visualization is the fastest way to learn what's in a dataset. Today we do it with **[Plotly](https://plotly.com/python/)**, so every chart is interactive: hover, zoom, click.

# Goals of this lecture
0. Know your data before you start.
0. Inspect data for faults.
0. Explore your data.
0. Find relationships, mathematically and visually.
0. Build interactive charts in Python with Plotly.

# Setup
First time? Run this in your terminal:

(`statsmodels` powers the trendlines later on.)

In [15]:
!pip install plotly statsmodels nbformat --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Our data management libraries
import pandas as pd
import numpy as np

# Plotly Express: one function call per chart, interactive by default
import plotly.express as px

# One line turns every df.plot() into an interactive Plotly chart
pd.options.plotting.backend = "plotly"

# Same category, same color, in every chart
px.defaults.color_discrete_map = {
    "Adelie": "#E07000", "Chinstrap": "#A020F0", "Gentoo": "#00A0A0",  # species
    "MALE": "#2E5EAA", "FEMALE": "#C2185B",                             # sex
}
px.defaults.color_discrete_sequence = ["#5B6B7F"]  # neutral slate for single-series charts

# Readable axis labels everywhere
px.defaults.labels = {
    "culmen_length_mm": "Culmen length (mm)",
    "culmen_depth_mm": "Culmen depth (mm)",
    "flipper_length_mm": "Flipper length (mm)",
    "body_mass_g": "Body mass (g)",
}

# First things first... load the data

In [3]:
df = pd.read_csv('data/penguins.csv')
df.head()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


Let's understand this dataset...
## WTH is a Culmen?
<img src='https://pbs.twimg.com/media/EaAXQn8U4AAoKUj.jpg:large' height=450 style="background-color:white;">

In [4]:
df['species'].value_counts()

species
Adelie       153
Gentoo       127
Chinstrap     68
Name: count, dtype: int64

<img src='https://allisonhorst.github.io/palmerpenguins/reference/figures/lter_penguins.png' height=450 style="background-color:white;">

<img src="https://www.gabemednick.com/post/penguin/featured_hu23a2ff6767279debab043a6c8f0a6157_878472_720x0_resize_lanczos_2.png" height=450>

![Palmer Archipelago, Antarctica](https://upload.wikimedia.org/wikipedia/commons/1/15/Palmer_archipelago%2C_Antarctica.jpg)

### Let's see how 'big' our data is by printing its shape.

In [5]:
print(df.shape)

(348, 7)


# Let's look at the description and info for our dataset.
* Why? It gives us a high-level summary of our data.

In [6]:
df.describe()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g
count,346.000000,346.000000,346.000000,346.000000
mean,44.136416,17.149133,201.447977,4219.916185
std,6.237503,1.970172,16.124117,817.796595
min,32.100000,13.100000,172.000000,2700.000000
25%,39.350000,15.600000,190.000000,3550.000000
50%,44.500000,17.300000,197.000000,4050.000000
75%,48.675000,18.700000,213.000000,4800.000000
max,100.200000,21.500000,349.000000,6891.000000


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 348 entries, 0 to 347
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            348 non-null    str    
 1   island             348 non-null    str    
 2   culmen_length_mm   346 non-null    float64
 3   culmen_depth_mm    346 non-null    float64
 4   flipper_length_mm  346 non-null    float64
 5   body_mass_g        346 non-null    float64
 6   sex                338 non-null    str    
dtypes: float64(4), str(3)
memory usage: 19.2 KB


# Let's check if we have any null values in our data.

In [8]:
df.isnull().sum()

species               0
island                0
culmen_length_mm      2
culmen_depth_mm       2
flipper_length_mm     2
body_mass_g           2
sex                  10
dtype: int64

### Remove, aka Drop, our null values

In [9]:
print(f'Shape before dropping NA: {df.shape}')

df = df.dropna()

# Sanity check
print(f'Shape after dropping NA: {df.shape}')
df.isnull().sum()

Shape before dropping NA: (348, 7)
Shape after dropping NA: (338, 7)


species              0
island               0
culmen_length_mm     0
culmen_depth_mm      0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64

### Instead of just dropping null values, you can fill them in creative ways.
[Here is a tutorial about that](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html)

```python
df = pd.read_csv('data/penguins.csv')
df['culmen_length_mm'] = df['culmen_length_mm'].fillna(df['culmen_length_mm'].median())
df.isnull().sum()
```

---

# Check for duplicate rows.

In [10]:
print(f'Number of duplicated rows: {df.duplicated().sum()}')
df[df.duplicated(keep=False)]

Number of duplicated rows: 3


,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
343,Gentoo,Biscoe,49.9,16.1,213.0,5400.0,MALE
344,Gentoo,Biscoe,49.9,16.1,213.0,5400.0,MALE
345,Gentoo,Biscoe,49.9,16.1,213.0,5400.0,MALE
346,Gentoo,Biscoe,49.9,16.1,213.0,5400.0,MALE


# Drop said duplicates...
`df.duplicated()` returns a boolean mask that's `True` when a row is a duplicate. How could we use it to remove the duplicated rows?

In [11]:
# Ask students to solve this one...
print(df.shape, 'before dropping dupes')

df = df.drop_duplicates()

print(df.shape, 'after dropping dupes')

(338, 7) before dropping dupes
(335, 7) after dropping dupes


# One more sneaky problem: check your categories
Nulls and dupes aren't the only faults. What values does `sex` take?

In [12]:
df['sex'].value_counts()

sex
MALE      169
FEMALE    165
.           1
Name: count, dtype: int64

In [13]:
# A '.' is not a sex. Drop it (or set it to NaN) before we color charts by sex.
df = df[df['sex'] != '.']
df['sex'].value_counts()

sex
MALE      169
FEMALE    165
Name: count, dtype: int64

# Plotting with pandas (powered by Plotly)
`df.plot(kind='scatter', x='COLUMN_NAME', y='COLUMN_NAME')`

Because we set `pd.options.plotting.backend = "plotly"`, these familiar pandas calls now return interactive Plotly charts. Kinds we can use:
```
- 'scatter' : scatter plot
- 'line'    : line plot (default)
- 'bar'     : vertical bar plot
- 'barh'    : horizontal bar plot
- 'hist'    : histogram
- 'box'     : boxplot
- 'area'    : area plot
```
**Try it:** hover over any point. Drag a box to zoom. Double-click to reset.

In [14]:
df.plot(
    kind='scatter',
    x='culmen_length_mm',
    y='flipper_length_mm',
    hover_data=['species', 'island', 'body_mass_g'],
    opacity=0.6,
)

# How do we remove this outlier...?
**Before you delete it, hover over it.** Which penguin is it? Do those numbers make sense for that species?

In [16]:
# Ask students to see if they can answer this.

select_condition = df.culmen_length_mm < 70

df = df[select_condition]

df.plot(
    kind='scatter',
    x='culmen_length_mm',
    y='flipper_length_mm',
    hover_data=['species', 'island', 'body_mass_g'],
    opacity=0.6,
)

# More plotting.

In [17]:
df.plot(kind='scatter',
        x='body_mass_g',
        y='flipper_length_mm',
        title='Body Mass vs Flipper Length')

# Plotting Group-by Objects

In [18]:
# Find the mean culmen_length_mm for each of the species
gb = df.groupby('species')
culmen_mean = gb['culmen_length_mm'].mean()

# Plot it
culmen_mean.plot(kind='barh',
                 title='Mean Culmen Length by Species',
                 labels={'value': 'Mean culmen length (mm)'}).update_layout(showlegend=False)  # hide the one-item legend

# Histograms are great for seeing what type of distribution the data has.

In [19]:
df['culmen_length_mm'].plot(kind='hist',
                            nbins=20,
                            title='Distribution of Culmen Length (mm)',
                            labels={'value': 'Culmen length (mm)'}).update_layout(showlegend=False)

In [20]:
df['body_mass_g'].plot(kind='hist', nbins=20, title='Body Mass (g)',
                       labels={'value': 'Body mass (g)'}).update_layout(showlegend=False)

**Question:** Does culmen length look like one bell curve? How many bumps do you see? (Hint: how many species are there?)

# Enter [Plotly Express](https://plotly.com/python/plotly-express/)
`df.plot()` is handy, but limited. Plotly Express (`px`) gives you every chart type with one function call:

`px.<chart_type>(df, x='COLUMN', y='COLUMN', color='COLUMN', ...)`

* Gallery of examples: https://plotly.com/python/plotly-express/

### Scatter plots with Plotly Express
* [Scatter plot reference](https://plotly.com/python/line-and-scatter/)

In [21]:
px.scatter(df,
           x='body_mass_g',
           y='culmen_length_mm',
           opacity=0.8,
           title="Relationship of Penguin's Culmen Length to Body Mass")

# Changing Axes

In [22]:
# Say you wanted to change the y-axis
px.scatter(df,
           x='body_mass_g',
           y='culmen_depth_mm',
           opacity=0.8,
           title="Relationship of Penguin's Body Mass to Culmen Depth")

## What if we wanted to change the color of the markers based on the species of penguin?
* Have students read the documentation to see if they can figure it out.
    * https://plotly.com/python/line-and-scatter/

In [23]:
px.scatter(df,
           x='body_mass_g',
           y='flipper_length_mm',
           color='species',
           opacity=0.7,
           title='Body Mass vs Flipper Length')

**Try it:** click a species in the legend to hide it. Double-click to isolate it.

In [24]:
# Color AND shape: still readable for colorblind viewers and in black-and-white printouts
px.scatter(df,
           x='body_mass_g',
           y='culmen_depth_mm',
           color='species',
           symbol='species',
           opacity=0.8,
           title="Relationship of Penguin's Body Mass to Culmen Depth")

### Every `px` function returns a `fig` object, which you can then customize.
Think of `fig` as the whole chart. Two methods do most of the work:
* `fig.update_layout(...)`: titles, legend, size, background ([layout reference](https://plotly.com/python/reference/layout/))
* `fig.update_traces(...)`: the marks themselves: marker size, opacity, line width

In [25]:
fig = px.scatter(df,
                 x='body_mass_g',
                 y='culmen_length_mm',
                 color='species',
                 hover_data=['island', 'sex'])

fig.update_layout(title="Relationship of Penguin's Body Mass to Culmen Length",
                  legend_title_text='Species',
                  template='plotly_white')
fig.update_traces(marker_size=9, opacity=0.75)

fig.show()

# The super scatter plot

In [26]:
# Marginal plots: the distribution of each axis, on the edges of the scatter
px.scatter(df,
           x='culmen_depth_mm',
           y='culmen_length_mm',
           color='species',
           marginal_x='histogram',
           marginal_y='box',
           opacity=0.8)

## Box & Whisker Plots
* A five-number summary turned into a picture:

1. Minimum (bottom whisker): smallest non-outlier value.
2. Q1 (25th percentile): 25% of the data lies below this point.
3. Median (50th percentile): the middle of the data.
4. Q3 (75th percentile): 75% of the data lies below this point.
5. Maximum (top whisker): largest non-outlier value.

**Try it:** hover over a box. Plotly prints the five-number summary for you.

In [27]:
px.box(df,
       x='species',
       y='culmen_length_mm',
       color='sex',
       title='Distribution of Culmen Length by Species and Sex')

## The violin plot
* It shows everything a box plot shows, and adds a kernel density estimation (KDE) on each side.
* In other words, it shows you the full shape of the distribution.

In [28]:
px.violin(df,
          x='species',
          y='culmen_length_mm',
          color='sex',
          box=True,        # draw the box plot inside the violin
          points='all',    # show every penguin as a dot
          title='Distribution of Culmen Length by Species and Sex')

## The catch-all plot function

In [29]:
# scatter_matrix shows every pair of numeric columns at once (seaborn calls this a pairplot)
fig = px.scatter_matrix(df,
                        dimensions=['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g'],
                        color='species',
                        opacity=0.6,
                        height=750)
fig.update_traces(diagonal_visible=False, marker_size=4)
fig.update_layout(dragmode='select')
fig.show()

**Try it:** drag a box around some points in any panel. The same penguins light up in *every* panel. This is called *linked brushing*, and a static chart can't do it.

## Plotting the line of best fit
* Add `trendline='ols'` to any `px.scatter` (OLS = ordinary least squares, i.e., linear regression).
    - No `color`: one line for all the data
    - With `color`: one line per group
* **Try it:** hover over a line to see its equation and R².

In [30]:
px.scatter(df,
           x='body_mass_g',
           y='culmen_length_mm',
           trendline='ols',
           opacity=0.6)

In [31]:
px.scatter(df,
           x='body_mass_g',
           y='culmen_length_mm',
           color='species',
           trendline='ols',
           opacity=0.6)

### Wait for it... a paradox
Let's look at culmen **length** vs culmen **depth**. First, all penguins together:

In [32]:
px.scatter(df,
           x='culmen_length_mm',
           y='culmen_depth_mm',
           trendline='ols',
           opacity=0.6,
           title='All penguins: longer culmen means shallower culmen?')

In [33]:
px.scatter(df,
           x='culmen_length_mm',
           y='culmen_depth_mm',
           color='species',
           trendline='ols',
           opacity=0.6,
           title='Split by species: the relationship flips')

Overall, the line slopes **down**. Within every species, it slopes **up**. This is **Simpson's paradox**: a trend that appears in the combined data reverses inside every group. Here, species is the hidden variable. Gentoos have long but shallow bills, which drags the combined line down.

**Lesson:** before trusting a trend, ask what group-level variable might be hiding.

# Finding Correlations in your data.
In the broadest sense, correlation is any statistical association, though it usually refers to how close two variables are to having a linear relationship.

[Learn more here](https://en.wikipedia.org/wiki/Correlation_and_dependence)

<img src='https://www.onlinemathlearning.com/image-files/correlation-coefficient.png' width=500>

In [34]:
# This function finds the correlation between all numerical values.
df.corr()  # why doesn't this work?

ValueError: could not convert string to float: 'Adelie'

In [36]:
# some columns are non-numeric
df.corr(numeric_only=True)

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g
culmen_length_mm,1.000000,-0.228626,0.653096,0.589451
culmen_depth_mm,-0.228626,1.000000,-0.577792,-0.472016
flipper_length_mm,0.653096,-0.577792,1.000000,0.872979
body_mass_g,0.589451,-0.472016,0.872979,1.000000


# Easily visualize your correlations with a heatmap.

In [37]:
corr = df.corr(numeric_only=True)

px.imshow(corr,
          text_auto='.2f',                  # print the values in each cell
          color_continuous_scale='RdBu_r',  # red = positive, blue = negative
          zmin=-1, zmax=1,                  # center the colors on 0, so white means "no correlation"
          title='Correlation between penguin measurements')

**Question:** Culmen length vs depth shows a *negative* correlation here. After the paradox above, do you believe it?

---
# Things a static chart can't do

## 1. Facets: one chart per group, in one line
`facet_col` splits the chart into side-by-side panels with shared axes.

In [38]:
px.scatter(df,
           x='culmen_length_mm',
           y='culmen_depth_mm',
           color='species',
           facet_col='island',
           opacity=0.7)

**Question:** Which island has all three species? Which species lives on only one island? Would you have spotted that in a single scatter plot?

## 2. Drill-down charts
A sunburst shows a hierarchy: island → species → sex. **Click a slice to zoom in. Click the center to zoom back out.**

In [39]:
px.sunburst(df,
            path=['island', 'species', 'sex'],
            color='species',
            title='Penguins by island, species, and sex')

## 3. Share it with someone who doesn't have Python
`fig.write_html()` saves a standalone web page. Anyone can open it in a browser, and it stays fully interactive.

In [ ]:
fig = px.scatter(df, x='body_mass_g', y='flipper_length_mm', color='species',
                 hover_data=['island', 'sex'], title='Palmer Penguins')
fig.write_html('penguins_chart.html')
print('Saved! Open penguins_chart.html in your browser.')

## 4. From charts to dashboards: [Dash](https://dash.plotly.com/)
Dash is Plotly's framework for turning these figures into a web app with dropdowns, sliders, and filters. It's all Python, with no JavaScript. A whole dashboard can be this small:

```python
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

app = Dash()
app.layout = html.Div([
    dcc.Dropdown(['Adelie', 'Chinstrap', 'Gentoo'], 'Adelie', id='species'),
    dcc.Graph(id='chart'),
])

@app.callback(Output('chart', 'figure'), Input('species', 'value'))
def update(species):
    return px.scatter(df[df['species'] == species], x='body_mass_g', y='flipper_length_mm')

app.run(debug=True)
```

---

# Main takeaways I want you to remember.
1. Use common sense: know what your data columns mean before anything else.
2. Always inspect your data for nulls, dupes, bad categories, and outliers. Hover before you delete.
3. `pd.options.plotting.backend = "plotly"` makes `df.plot()` interactive. `px` gives you every chart in one call, and it returns a `fig` you customize with `update_layout` / `update_traces`.
4. Interactivity is for *exploring*: hover, zoom, legend clicks, linked brushing, facets.
5. Check for hidden groups (Simpson's paradox) before you trust a trend.


##### End of this lecture.

---